## Imports

In [104]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage
import asyncio
import os

In [105]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__
        sys.stdout = sys.__stdout__

## Prompts

In [106]:
SUPERVISOR_SYSTEM_PROMPT = """
Você é o Supervisor do Assistente de Programação Concorrente. 
Sua função é coordenar os subagentes para garantir uma resposta pedagógica e segura.

## Fluxo de Decisão:
1. **Segurança Primeiro**: Sempre comece chamando o `call_policy_subagent`. 
   - Se ele recusar a pergunta (ex: Pedido de resposta de exercício), encerre com a mensagem de recusa.
2. **Busca de Conhecimento**: Se a pergunta for válida, chame o `call_retriver_subagent`.
3. **Geração de Resposta**: Com os documentos em mãos, chame o `call_qa_subagent` para formular a resposta.
4. **Verificação (Self-Check)**: Envie a resposta do QA para o `call_selfcheck_subagent`.
   - Se o veredito for 'REQUER_REBUSCA', tente chamar o retriever mais uma vez com termos diferentes.
   - Se for 'APROVADO', siga para a automação.
5. **Automação (Obrigatório)**: Após a aprovação da resposta, chame o `call_automation_subagent` para registrar o log no sistema de arquivos para o professor.
6. **Entrega Final (Saída)**: APÓS a confirmação de sucesso da automação, entregue a resposta final (aprovada no passo 4) ao aluno e declare o processo como CONCLUÍDO.

## Regras Críticas:
- Nunca forneça código completo que resolva exercícios.
- Sempre garanta que o log foi salvo via automação antes de entregar a resposta final ao aluno.
- Para entregar a resposta final, use o conteúdo exato validado pelo QA, sem adicionar novas informações.
"""

In [107]:
RETRIEVER_SYSTEM_PROMPT = """
Você é o Agente Recuperador de um assistente educacional de Programação Concorrente.
Sua única responsabilidade é buscar trechos relevantes nos documentos indexados da disciplina.

## Comportamento esperado

Dado uma query do aluno, você deve:
1. Usar a ferramenta `search_for_information_on_indexed_documents` para buscar passagens relevantes.
2. Retornar os trechos encontrados com suas respectivas fontes e páginas, SEM interpretar ou responder.
3. Se nenhum trecho relevante for encontrado, retornar explicitamente: "NENHUMA_EVIDENCIA_ENCONTRADA".

## Formato de saída

Retorne os trechos no seguinte formato:

[TRECHO 1]
Fonte: <nome do documento> | Página: <número>
Conteúdo: <texto do trecho>

[TRECHO 2]
...

## Restrições
- NÃO responda à pergunta do aluno.
- NÃO adicione interpretações, resumos ou opiniões.
- NÃO invente trechos. Retorne APENAS o que foi encontrado nas ferramentas.
- Se a busca retornar resultados irrelevantes, informe: "TRECHOS_IRRELEVANTES".
"""

In [108]:
POLICY_SYSTEM_PROMPT = """
Você é o Agente de Política de um assistente educacional de Programação Concorrente.
Você recebe a pergunta do aluno e os trechos recuperados dos documentos.

Avalie e responda em linguagem natural com dois parágrafos curtos:

1. Se a resposta pode ou não ser gerada — justificando brevemente por que.
   O assistente NÃO pode resolver exercícios, completar código de tarefas
   ou confirmar respostas avaliativas. Pode explicar conceitos, dar exemplos
   genéricos e sugerir abordagens.

2. Se aplicável, escreva um disclaimer para ser incluído na resposta final.
   Use disclaimer quando a dúvida envolver: primitivas de sincronização,
   deadlock/livelock, ou exemplos de código (marque como ilustrativos).
   Se não precisar de disclaimer, escreva: "Sem disclaimer necessário."
"""

In [109]:
QA_SYSTEM_PROMPT = """
Você é o Agente Respondedor de um assistente educacional de Programação Concorrente.
Seu papel é responder dúvidas conceituais dos alunos de forma didática, clara e SEMPRE
com citações dos documentos recuperados.

## Princípios pedagógicos

- Priorize a compreensão: explique o "porquê", não apenas o "o quê".
- Use analogias simples quando o conceito for abstrato.
- Guie o raciocínio do aluno com perguntas retóricas quando apropriado.
- Nunca entregue soluções prontas — sugira caminhos e conceitos relevantes.

## Formato obrigatório da resposta

1. **Resposta principal**: explicação do conceito em linguagem acessível.
2. **Exemplo ilustrativo** (se aplicável): trecho de pseudocódigo ou analogia.
3. **Fontes utilizadas**: cite OBRIGATORIAMENTE os trechos recuperados no formato:
   > 📄 *<nome do documento>*, p. <página>: "<trecho relevante>"
4. **Disclaimer** (se aplicável): inclua os avisos indicados pelo Policy Agent.
5. **Próximos passos sugeridos**: indique ao aluno o que estudar em seguida.

## Restrições
- NUNCA responda sem citar ao menos 1 fonte dos documentos recuperados.
- Se os documentos não cobrirem a dúvida, diga explicitamente:
  "Não encontrei cobertura suficiente nos materiais da disciplina para esta dúvida.
   Recomendo consultar [sugestão de recurso externo]."
- Não invente citações. Use apenas os trechos fornecidos pelo Retriever Agent.
"""

In [110]:
SELFCHECK_SYSTEM_PROMPT = """
Você é o Agente de Verificação de um assistente educacional de Programação Concorrente.
Você recebe a resposta gerada e os trechos dos documentos que a embasaram.

Analise em linguagem natural:
- As afirmações centrais da resposta têm suporte nos trechos fornecidos?
- Alguma afirmação foi inventada ou distorcida além do que os documentos dizem?

Conclua com uma das frases exatas abaixo, seguida de uma justificativa breve:

APROVADO — todas as afirmações estão suportadas pelos documentos.
REQUER_REBUSCA — há afirmações sem suporte; sugira uma query mais específica.
RECUSADO — a resposta contém afirmações contraditórias ou sem qualquer evidência.

Inferências razoáveis a partir dos documentos são aceitáveis se estiverem
claramente marcadas como tal na resposta.

Sua saída deve ser um JSON (ou formato estruturado) contendo:
veredito: [APROVADO, RE-BUSCA, RECUSADO]
motivo: 'Explicação curta'

Critério de Ouro: Se o Respondedor usou um conhecimento geral de LLM que NÃO estava nos documentos recuperados, marque como RE-BUSCA. O sistema deve ser estritamente baseado no corpus da disciplina.
"""

In [134]:
LOG_DIR = os.path.abspath("./logs_professor")

In [135]:
AUTOMATION_SYSTEM_PROMPT = f"""
Você é um assistente de auditoria pedagógica.
Sua tarefa é salvar logs de interações.
O diretório permitido é: {LOG_DIR}
SEMPRE use o caminho COMPLETO: {LOG_DIR}/log_interacao.txt
NUNCA use caminhos relativos ou apenas o nome do arquivo.
"""

## Inicializando os agentes

In [ ]:
#os.environ["GOOGLE_API_KEY"] = "AIzaSyAIcyrS1V_lHANkvCW1lv4yfT0mvznsis"
from dotenv import load_dotenv
load_dotenv(override=True)
load_dotenv()
# model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",max_retries=5,timeout=120)
model = ChatGroq(
    model="llama-3.3-70b-versatile", temperature=0,max_retries=5,timeout=120
)


In [114]:
DIRETORIO_BANCO = "./db_chroma"

In [115]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

vector_store = Chroma(
        persist_directory=os.path.abspath(DIRETORIO_BANCO),
        embedding_function=embeddings,
        collection_name="concorrencia_db"
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

teste = vector_store.similarity_search("threads", k=1)
print(f"Resultado do teste: {teste}")

@tool("search_for_information_on_indexed_documents")
def search_for_information_on_indexed_documents(query: str) -> str:
    """Busca informações nos documentos PDF indexados para responder ao aluno."""
    try:
        # Tenta recuperar os documentos
        documentos_encontrados = retriever.invoke(query)
        
        if not documentos_encontrados:
            return "AVISO: Nenhum trecho relevante foi encontrado nos livros sobre este assunto."

        resultados = []
        for doc in documentos_encontrados:
            pagina = doc.metadata.get('page', 'Desconhecida')
            origem = doc.metadata.get('source', 'Desconhecida')
            resultados.append(f"FONTE: {origem} (Pág {pagina})\nCONTEÚDO: {doc.page_content}\n---")

        return "\n\n".join(resultados)
    except Exception as e:
        return f"Erro ao acessar o banco de dados: {str(e)}"

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Resultado do teste: [Document(metadata={'keywords': '', 'total_pages': 291, 'author': '', 'page_label': '191', 'producer': 'dvips + GPL Ghostscript 9.10', 'title': '', 'source': './pdfs/LittleBookOfSemaphores.pdf', 'moddate': '2016-06-17T11:50:39-04:00', 'subject': '', 'creator': 'LaTeX with hyperref package', 'creationdate': '2016-06-17T11:50:39-04:00', 'page': 202}, page_content='thread is doing work that seems, logically, to belong to the w aiting threads.\nA drawback of this approach is that is it a little more diﬃcult to conﬁrm\nthat the state is being updated correctly.')]


In [116]:
print(retriever.invoke("concurrency"))

[Document(metadata={'subject': '', 'page_label': '297', 'creationdate': 'D:20140526154906', 'keywords': '', 'author': '', 'producer': 'dvips + AFPL Ghostscript 8.54', 'moddate': 'D:20140526154906', 'creator': 'LaTeX with hyperref package', 'page': 296, 'title': '', 'total_pages': 643, 'source': './pdfs/Three Easy Pieces.pdf'}, page_content='25\nA Dialogue on Concurrency\nProfessor: And thus we reach the second of our three pillars of operating sys-\ntems: concurrency.\nStudent: I thought there were four pillars...?\nProfessor: Nope, that was in an older version of the book.\nStudent: Umm... OK. So what is concurrency, oh wonderful professor?\nProfessor: Well, imagine we have a peach –\nStudent: (interrupting) Peaches again! What is it with you and peache s?\nProfessor: Ever read T.S. Eliot? The Love Song of J. Alfred Prufrock, “Do I dare\nto eat a peach”, and all that fun stuff?\nStudent: Oh yes! In English class in high school. Great stuff! I really liked the\npart where –\nProfessor:

In [ ]:



# client = MultiServerMCPClient(
#     {
#         "local_server": {
#             "transport": "sse",
#             "url":"http://localhost:8000/sse"
#         }
#     }
# )

client = MultiServerMCPClient(
        {
            "logs_manager": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", LOG_DIR],
                "transport": "stdio" 
            }
        }
    )




In [118]:
mcp_tools = await client.get_tools()

# get resources

# get prompts
# prompt = await client.get_prompt("local_server", "prompt")
# prompt = prompt[0].content


In [136]:
agent_policy = create_agent(model=model, system_prompt=POLICY_SYSTEM_PROMPT)
agent_retriver = create_agent(model=model, tools=[search_for_information_on_indexed_documents], system_prompt=RETRIEVER_SYSTEM_PROMPT)
agent_qa = create_agent(model=model, system_prompt=QA_SYSTEM_PROMPT)
agent_check = create_agent(model=model, system_prompt=SELFCHECK_SYSTEM_PROMPT)
agent_automation = create_agent(model=model, tools=mcp_tools, system_prompt=AUTOMATION_SYSTEM_PROMPT)

In [137]:

@tool
async def call_policy_subagent(query: str) -> list:
  """Chama o subagente 1 que impede do usuário fazer perguntas que violam as regras impostas pelo sistema"""
  # await asyncio.sleep(3)
  response =  await agent_policy.ainvoke({"messages": [HumanMessage(content=f"Me diga se a pergunta desse usuário viola alguma das regras do sistema. {query}")]})
  return response["messages"][-1].content


@tool
async def call_retriver_subagent(query: str) -> list:
  """Chama o subagente 2 que busca os documentos mais relevantes de acordo com a query"""
  # await asyncio.sleep(3)
  response = await agent_retriver.ainvoke({"messages": [HumanMessage(content=f"Me retorne os documentos mais relevantes de acordo com essa query {query}")]})
  return response["messages"][-1].content



@tool
async def call_qa_subagent(query: str, docs) -> list:
  """Chama o subagente 3 que estrutura uma resposta de acordo com a pergunta e os documentos passados"""
  # await asyncio.sleep(3)
  response = await agent_qa.ainvoke({"messages": [HumanMessage(content=f"Responda a pergunta com base nos documentos fornecidos. Pergunta: {query}. Documentos recuperados: {docs}")]})
  return response["messages"][-1].content

@tool
async def call_selfcheck_subagent(query: str, docs, response: str) -> list:
  """Chama o subagente 4 que valida a resposta do usuário com base em regras pré definidas e decide se é válida, necessita de re-busca ou retornar uma mensagem de pergunta inválida"""
  # await asyncio.sleep(3)
  response_agent = await agent_check.ainvoke({"messages": [HumanMessage(content=f"Valide a resposta do qa com base nas regras pré-definidas. Pergunta: {query}. Resposta: {response}. Documentos recuperados: {docs}")]})
  return response_agent["messages"][-1].content

@tool
async def call_logging_automation(pergunta: str, resposta: str, veredito: str) -> str:
    """Registra a interação aluno-IA usando logs locais."""
    
    # O SEGREDO: Use apenas o nome do arquivo. 
    # O MCP já sabe que deve salvar dentro de /home/.../logs_professor/
    filename = "log_interacao.txt" 
    
    log_content = f"Pergunta: {pergunta}\nResposta: {resposta}\nVeredito: {veredito}"
    
    # Instrução ultra-específica para o Llama 3 não inventar caminhos
    instrucao = f"Escreva EXATAMENTE no arquivo '{filename}' o seguinte conteúdo: {log_content}"
    
    await agent_automation.ainvoke({"messages": [HumanMessage(content=instrucao)]})
    return "Log registrado com sucesso."


In [138]:
agent_supervisor = create_agent(model=model, tools=[call_retriver_subagent, call_policy_subagent, call_qa_subagent, call_selfcheck_subagent, call_logging_automation], system_prompt=SUPERVISOR_SYSTEM_PROMPT)


In [139]:
from langchain_core.messages import HumanMessage  

config = {"configurable": {"thread_id": "teste_novo_caminho"}}

response = await agent_supervisor.ainvoke(
    {"messages": [HumanMessage(content="What is concurrency in the context of Programming?")]},
    config=config
)

In [140]:
response

{'messages': [HumanMessage(content='What is concurrency in the context of Programming?', additional_kwargs={}, response_metadata={}, id='e5e26f64-76b3-4d5b-8259-4f50ab03a881'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'png5v7ck4', 'function': {'arguments': '{"query":"What is concurrency in the context of Programming?"}', 'name': 'call_policy_subagent'}, 'type': 'function'}, {'id': '32epqmyce', 'function': {'arguments': '{"query":"What is concurrency in the context of Programming?"}', 'name': 'call_retriver_subagent'}, 'type': 'function'}, {'id': 'a7na90db8', 'function': {'arguments': '{"docs":{"args":[{"query":"What is concurrency in the context of Programming?"}],"function_name":"call_retriver_subagent"},"query":"What is concurrency in the context of Programming?"}', 'name': 'call_qa_subagent'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 328, 'prompt_tokens': 1099, 'total_tokens': 1427, 'completion_time': 0.642316354, 'comple

In [100]:
# Verifique se está assim:
LOG_DIR = os.path.abspath("./logs_professor")
# ...
print(LOG_DIR)

/home/jardel/Documentos/UFCG/llm/projeto_final/Projeto-LLM/logs_professor
